# Caching

In [2]:
# ==========================================
# MASTER SWITCHES
# True  = Load saved data/models if they exist
# False = Force process and train from scratch
USE_CACHE_DATA = True
USE_CACHE_DEEPBIND = False
USE_CACHE_IDEEPS = True
USE_CACHE_ATTENTION = False
# ==========================================

# import os
# import shutil

# input_dir = "/kaggle/input"
# target_dir = "/kaggle/working/ideeps_output"
# os.makedirs(target_dir, exist_ok=True)

# print("Scanning every single folder in Kaggle's input directory...")
# data_root = None

# # Hunt down the 'cache' and 'models' folders anywhere in the input directory
# for root, dirs, files in os.walk(input_dir):
#     if "cache" in dirs and "models" in dirs:
#         data_root = root
#         break

# if data_root:
#     print(f" Found your exact data location: {data_root}")
#     print("Copying files over safely...")
    
#     for item in os.listdir(data_root):
#         s = os.path.join(data_root, item)
#         d = os.path.join(target_dir, item)
#         if os.path.isdir(s):
#             shutil.copytree(s, d, dirs_exist_ok=True)
#         else:
#             shutil.copy2(s, d)
#     print(" Data successfully restored! You can now delete this cell and run Cell 1.")
# else:
#     print("Still couldn't find it. Here is exactly what is inside /kaggle/input right now:")
#     for item in os.listdir(input_dir):
#         print(f" - {item}")

Scanning every single folder in Kaggle's input directory...
 Found your exact data location: /kaggle/input/datasets/karankeer/clean-ideeps-data
Copying files over safely...
 Data successfully restored! You can now delete this cell and run Cell 1.


# Intialisation

In [6]:
!apt-get update -y && apt-get install -y libipc-system-simple-perl

import gzip
import itertools
import json
import os
import random
import shutil
import stat
import subprocess
import tempfile
import zipfile
import time
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- Configuration & Globals ---
PROJECT_ROOT = Path("/kaggle/working/ideeps_output")
DEFAULT_DATASETS = ["17_ICLIP_HNRNPC_hg19", "10_PARCLIP_ELAVL1A_hg19", "28_ICLIP_TIA1_hg19"]

config = {
    "seed": 629, "raw_sequence_length": 101, "sequence_length": 101,
    "conv_motif_length": 10, "conv_pad": 5, "conv_input_length": 111,
    "batch_size": 50, "max_epochs": 30, "early_stopping_patience": 5,
    "learning_rate": 1e-3, "structure_alphabet": "FTIHMS", "cache_version": "paperstyle_v1",
}
BASE_DATA_URL = "https://raw.githubusercontent.com/xypan1232/iDeepS/master/datasets/clip"
RNA_SHAPES_URL = "https://raw.githubusercontent.com/xypan1232/iDeepS/master/RNAshapes"
GRAPHPROT_ZIP_URL = "https://github.com/dmaticzka/GraphProt/archive/refs/heads/master.zip"
NUC_TO_INDEX = {"A": 0, "C": 1, "G": 2, "U": 3}

# --- Utilities ---
def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def download_file_if_missing(url: str, dest_path: Path) -> Path:
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if not dest_path.exists():
        print(f"Downloading {url} -> {dest_path}")
        urlretrieve(url, dest_path)
    return dest_path

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def print_header(title: str) -> None:
    print("\n" + "=" * 90); print(title); print("=" * 90)

def get_positive_probabilities(model, X: np.ndarray) -> np.ndarray:
    if hasattr(model, "predict_proba"): return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = np.clip(model.decision_function(X), -30.0, 30.0)
        return 1.0 / (1.0 + np.exp(-scores))
    raise ValueError("Model does not support probability-like outputs")

def compute_binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob).astype(float) >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "y_pred": y_pred,
    }

def save_roc_plot(y_true, y_prob, out_path, title):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_true, y_prob):.4f}", linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.savefig(out_path, dpi=200); plt.close()

def save_confusion_matrix_plot(cm, out_path, title):
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted label"); plt.ylabel("True label"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_path, dpi=200); plt.close()

def save_training_history_plot(history, out_path, title):
    hist = history.history
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(hist.get("loss", []), label="train_loss")
    plt.plot(hist.get("val_loss", []), label="val_loss")
    plt.title(f"{title} - Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.subplot(1, 2, 2)
    if "auc" in hist: plt.plot(hist.get("auc", []), label="train_auc")
    if "val_auc" in hist: plt.plot(hist.get("val_auc", []), label="val_auc")
    plt.title(f"{title} - AUC"); plt.xlabel("Epoch"); plt.ylabel("AUC"); plt.legend()
    plt.tight_layout(); plt.savefig(out_path, dpi=200); plt.close()

def make_paths(project_root: Path) -> dict:
    cache_dir = ensure_dir(project_root / "cache")
    results_dir = ensure_dir(project_root / "results")
    figures_dir = ensure_dir(project_root / "figures")
    models_dir = ensure_dir(project_root / "models")
    predictions_dir = ensure_dir(results_dir / "predictions")
    return {
        "project_root": project_root, "cache_dir": cache_dir, "models_dir": models_dir,
        "figures_dir": figures_dir, "results_dir": results_dir,
        "toolchain_dir": ensure_dir(project_root / "toolchain"),
        "raw_dir": ensure_dir(cache_dir / "raw"),
        "processed_dir": ensure_dir(cache_dir / "processed"),
        "predictions_dir": predictions_dir,
        "summary_csv": results_dir / "summary_metrics.csv",
        "oli_fig_dir": ensure_dir(figures_dir / "paper_baseline_oli"),
        "oli_pred_dir": ensure_dir(predictions_dir / "paper_baseline_oli"),
        "graphprot_model_dir": ensure_dir(models_dir / "paper_baseline_graphprot"),
        "graphprot_fig_dir": ensure_dir(figures_dir / "paper_baseline_graphprot"),
        "graphprot_pred_dir": ensure_dir(predictions_dir / "paper_baseline_graphprot"),
        "graphprot_input_dir": ensure_dir(cache_dir / "graphprot_inputs"),
        "main_model_dir": ensure_dir(models_dir / "ideeps_like_model"),
        "main_fig_dir": ensure_dir(figures_dir / "ideeps_like_model"),
        "main_pred_dir": ensure_dir(predictions_dir / "ideeps_like_model"),
        "final_comparison_fig_dir": ensure_dir(figures_dir / "final_comparison"),
    }

# --- Toolchain & Preprocessing logic ---
def ensure_rnashapes_binary(paths: dict) -> Path:
    rnashapes_path = paths["toolchain_dir"] / "RNAshapes"
    download_file_if_missing(RNA_SHAPES_URL, rnashapes_path)
    rnashapes_path.chmod(rnashapes_path.stat().st_mode | 0o111)
    return rnashapes_path

def ensure_graphprot(paths: dict) -> Path:
    graphprot_root = paths["toolchain_dir"] / "GraphProt"
    
    # ONLY download and compile if the folder doesn't exist yet
    if not graphprot_root.exists():
        graphprot_zip = paths["toolchain_dir"] / "GraphProt.zip"
        download_file_if_missing(GRAPHPROT_ZIP_URL, graphprot_zip)
        with tempfile.TemporaryDirectory() as tmpdir:
            with zipfile.ZipFile(graphprot_zip, "r") as zf:
                zf.extractall(tmpdir)
            shutil.move(str(Path(tmpdir) / "GraphProt-master"), str(graphprot_root))

        # --- THE REAL FIX: Natively Recompile EDeN ---
        print("Compiling EDeN C++ binary natively for Kaggle (This takes ~1 minute)...")
        subprocess.run(["bash", "recompile_EDeN.sh"], cwd=str(graphprot_root), capture_output=False)

    # Force executable permissions
    graphprot_pl = graphprot_root / "GraphProt.pl"
    if graphprot_pl.exists(): 
        graphprot_pl.chmod(graphprot_pl.stat().st_mode | 0o111)
        
    eden_bin = graphprot_root / "EDeN" / "EDeN"
    if eden_bin.exists(): 
        eden_bin.chmod(eden_bin.stat().st_mode | 0o111)

    return graphprot_root
    
def verify_graphprot_dependencies():
    perl_check = subprocess.run(["perl", "-MIPC::System::Simple", "-e", "1"], capture_output=True, text=True)
    if perl_check.returncode != 0: raise RuntimeError("Missing Perl module IPC::System::Simple.")

def make_graphprot_env(paths: dict) -> dict:
    env = os.environ.copy()
    graphprot_root = ensure_graphprot(paths)
    rnashapes_binary = ensure_rnashapes_binary(paths)
    env["PATH"] = f"{graphprot_root}:{graphprot_root / 'bin'}:{rnashapes_binary.parent}:{env.get('PATH', '')}"
    return env

def dataset_urls(dataset_name: str):
    return f"{BASE_DATA_URL}/{dataset_name}/30000/training_sample_0/sequences.fa.gz", f"{BASE_DATA_URL}/{dataset_name}/30000/test_sample_0/sequences.fa.gz"

def download_dataset_files(paths, dataset_name):
    dataset_dir = ensure_dir(paths["raw_dir"] / dataset_name)
    train_url, test_url = dataset_urls(dataset_name)
    return download_file_if_missing(train_url, dataset_dir / "training_sequences.fa.gz"), download_file_if_missing(test_url, dataset_dir / "test_sequences.fa.gz")

def parse_fasta_gz(file_path):
    records = []
    with gzip.open(file_path, "rt") as fp:
        current_header, current_seq_parts = None, []
        for line in fp:
            line = line.strip()
            if not line: continue
            if line.startswith(">"):
                if current_header is not None:
                    records.append({"header": current_header, "sequence": "".join(current_seq_parts).upper().replace("T", "U"), "label": int(current_header.split("class:")[-1])})
                current_header, current_seq_parts = line[1:], []
            else: current_seq_parts.append(line)
        if current_header is not None:
            records.append({"header": current_header, "sequence": "".join(current_seq_parts).upper().replace("T", "U"), "label": int(current_header.split("class:")[-1])})
    return records

def split_train_validation(records, val_size, seed):
    sequences, labels, headers = [r["sequence"] for r in records], np.array([r["label"] for r in records], dtype=np.int32), [r["header"] for r in records]
    train_idx, val_idx = train_test_split(np.arange(len(records)), test_size=val_size, random_state=seed, stratify=labels, shuffle=True)
    return {"headers": [headers[i] for i in train_idx], "sequences": [sequences[i] for i in train_idx], "labels": labels[train_idx]}, {"headers": [headers[i] for i in val_idx], "sequences": [sequences[i] for i in val_idx], "labels": labels[val_idx]}

def package_test(records):
    return {"headers": [r["header"] for r in records], "sequences": [r["sequence"] for r in records], "labels": np.array([r["label"] for r in records], dtype=np.int32)}

# --- THE KEYERROR 'N' FIX ---
def get_rna_seq_convolutional_array(seq, motif_len):
    seq = seq.upper().replace("T", "U")
    half_len = motif_len // 2
    arr = np.zeros((len(seq) + (half_len * 2), 4), dtype=np.float32)
    arr[:half_len, :] = 0.25; arr[len(seq) + half_len:, :] = 0.25
    for i, ch in enumerate(seq):
        if ch in NUC_TO_INDEX:
            arr[i + half_len, NUC_TO_INDEX[ch]] = 1.0 
        else:
            arr[i + half_len, :] = 0.25
    return arr

def encode_sequence_list(sequences, target_length, config):
    encoded = []
    for seq in sequences:
        seq = seq[:target_length]
        if len(seq) < target_length: seq += "N" * (target_length - len(seq))
        encoded.append(get_rna_seq_convolutional_array(seq, config["conv_motif_length"]))
    return np.stack(encoded, axis=0)

def parse_dot_bracket_pairs(dot_bracket):
    stack, pair_map = [], {}
    for idx, ch in enumerate(dot_bracket):
        if ch == "(": stack.append(idx)
        elif ch == ")": 
            left = stack.pop()
            pair_map[left] = idx; pair_map[idx] = left
    return pair_map

def find_top_level_child_intervals(dot_bracket, pair_map, region_start, region_end):
    children, pos = [], region_start
    while pos < region_end:
        if dot_bracket[pos] == "(": child_end = pair_map[pos]; children.append((pos, child_end)); pos = child_end + 1
        else: pos += 1
    return children

def fill_unpaired(labels, start, end, symbol):
    for i in range(start, end):
        if labels[i] == "?": labels[i] = symbol

def annotate_dot_bracket_with_six_letters(dot_bracket):
    pair_map, n = parse_dot_bracket_pairs(dot_bracket), len(dot_bracket)
    labels = ["?"] * n
    for i, ch in enumerate(dot_bracket):
        if ch in "()": labels[i] = "S"
    def annotate_loop_region(region_start, region_end, is_root=False):
        children = find_top_level_child_intervals(dot_bracket, pair_map, region_start, region_end)
        if is_root:
            if not children: fill_unpaired(labels, region_start, region_end, "F")
            else:
                fill_unpaired(labels, region_start, children[0][0], "F")
                for (_, prev_end), (next_start, _) in zip(children, children[1:]): fill_unpaired(labels, prev_end + 1, next_start, "M")
                fill_unpaired(labels, children[-1][1] + 1, region_end, "T")
        else:
            if not children: fill_unpaired(labels, region_start, region_end, "H")
            elif len(children) == 1:
                fill_unpaired(labels, region_start, children[0][0], "I"); fill_unpaired(labels, children[0][1] + 1, region_end, "I")
            else:
                prev = region_start
                for cs, ce in children: fill_unpaired(labels, prev, cs, "M"); prev = ce + 1
                fill_unpaired(labels, prev, region_end, "M")
        for cs, ce in children: annotate_loop_region(cs + 1, ce, is_root=False)
    annotate_loop_region(0, n, is_root=True)
    for i in range(n):
        if labels[i] == "?": labels[i] = "F"
    return "".join(labels)

def run_rnashape(sequence, paths):
    result = subprocess.run([str(ensure_rnashapes_binary(paths)), "-t", "5", "-c", "10", "-#", "1"], input=sequence.upper().replace("T", "U") + "\n", text=True, capture_output=True, check=True)
    lines = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    return annotate_dot_bracket_with_six_letters((lines[-2] if "configured to print" in lines[-1] else lines[1]).split()[1])

def encode_rnashape_structure(shape_string, config):
    structure_to_index = {ch: idx for idx, ch in enumerate(config["structure_alphabet"])}
    half_len, alpha_size = config["conv_motif_length"] // 2, len(config["structure_alphabet"])
    arr = np.zeros((len(shape_string) + (half_len * 2), alpha_size), dtype=np.float32)
    arr[:half_len, :] = 1.0 / alpha_size; arr[len(shape_string) + half_len:, :] = 1.0 / alpha_size
    for i, ch in enumerate(shape_string): arr[i + half_len, structure_to_index.get(ch, slice(None))] = 1.0 if ch in structure_to_index else 1.0 / alpha_size
    return arr

def encode_structure_list_with_rnashapes(sequences, headers, cache_dir, paths, config):
    ensure_dir(cache_dir)
    structure_gz_path = cache_dir / "structure.gz"
    cached_map = {}
    if structure_gz_path.exists():
        with gzip.open(structure_gz_path, "rt") as fp:
            curr_header = None
            for line in fp:
                line = line.strip()
                if line.startswith(">"): curr_header = line[1:]
                elif line and curr_header: cached_map[curr_header] = line
                
    structures, arrays = [], []
    total_seqs = len(sequences)
    
    for idx, (header, seq) in enumerate(zip(headers, sequences)):
        # --- THE RNAshapes HEARTBEAT FIX ---
        if idx % 1000 == 0 or idx == total_seqs - 1:
            print(f"  -> RNAshapes processing: {idx}/{total_seqs} sequences completed...")
            
        shape_string = cached_map.get(header) or run_rnashape(seq, paths)
        cached_map[header] = shape_string
        structures.append(shape_string)
        arrays.append(encode_rnashape_structure(shape_string, config))
        
    with gzip.open(structure_gz_path, "wt") as fw:
        for header, shape_string in zip(headers, structures): fw.write(f">{header}\n{shape_string}\n")
    return np.stack(arrays, axis=0), structures

def build_processed_dataset(dataset_name, paths, config):
    processed_path = paths["processed_dir"] / f"{dataset_name}_{config['cache_version']}.npz"
    if processed_path.exists():
        print(f"Loading cached dataset for {dataset_name}")
        data = np.load(processed_path, allow_pickle=True)
        return {key: data[key] for key in data.files}
    print(f"Preparing dataset: {dataset_name}")
    train_records = parse_fasta_gz(download_dataset_files(paths, dataset_name)[0])
    test_records = parse_fasta_gz(download_dataset_files(paths, dataset_name)[1])
    train_split, val_split = split_train_validation(train_records, val_size=0.2, seed=config["seed"])
    test_split = package_test(test_records)
    
    x_struct_train, struct_train_strings = encode_structure_list_with_rnashapes(train_split["sequences"], train_split["headers"], paths["processed_dir"] / f"{dataset_name}_structures/train", paths, config)
    x_struct_val, struct_val_strings = encode_structure_list_with_rnashapes(val_split["sequences"], val_split["headers"], paths["processed_dir"] / f"{dataset_name}_structures/val", paths, config)
    x_struct_test, struct_test_strings = encode_structure_list_with_rnashapes(test_split["sequences"], test_split["headers"], paths["processed_dir"] / f"{dataset_name}_structures/test", paths, config)
    
    result = {
        "x_seq_train": encode_sequence_list(train_split["sequences"], config["sequence_length"], config),
        "x_seq_val": encode_sequence_list(val_split["sequences"], config["sequence_length"], config),
        "x_seq_test": encode_sequence_list(test_split["sequences"], config["sequence_length"], config),
        "x_struct_train": x_struct_train, "x_struct_val": x_struct_val, "x_struct_test": x_struct_test,
        "y_train": train_split["labels"], "y_val": val_split["labels"], "y_test": test_split["labels"],
        "train_headers": train_split["headers"], "val_headers": val_split["headers"], "test_headers": test_split["headers"],
        "struct_train_strings": struct_train_strings, "struct_val_strings": struct_val_strings, "struct_test_strings": struct_test_strings,
        "train_sequences": train_split["sequences"], "val_sequences": val_split["sequences"], "test_sequences": test_split["sequences"],
    }
    np.savez_compressed(processed_path, **{k: (np.array(v, dtype=object) if "header" in k or "string" in k else v) for k, v in result.items()})
    return result

# --- SYNTHETIC DATASET GENERATION ---
def build_synthetic_dataset(paths, config):
    synthetic_cache_path = paths["processed_dir"] / f"synthetic_dataset_{config['cache_version']}.npz"
    if synthetic_cache_path.exists():
        print(f"Loading cached synthetic dataset from {synthetic_cache_path}")
        data = np.load(synthetic_cache_path, allow_pickle=True)
        return {key: data[key] for key in data.files}

    print("Building synthetic dataset...")
    rng = random.Random(config["seed"])
    motif, seq_len = "UGCAUG", config["sequence_length"]
    records = []

    def random_sequence_without_motif(length, motif, rng):
        for _ in range(1000):
            seq = "".join(rng.choice(["A", "C", "G", "U"]) for _ in range(length))
            if motif not in seq: return seq
        raise RuntimeError("Could not generate sequence without motif")

    for _ in range(6000): 
        start = rng.randint(0, seq_len - len(motif))
        seq = random_sequence_without_motif(seq_len, motif, rng)
        records.append({"sequence": seq[:start] + motif + seq[start+len(motif):], "structure": ("F"*seq_len)[:start] + ("H"*len(motif)) + ("F"*seq_len)[start+len(motif):], "label": 1})
    for _ in range(3000): 
        start = rng.randint(0, seq_len - len(motif))
        seq = random_sequence_without_motif(seq_len, motif, rng)
        records.append({"sequence": seq[:start] + motif + seq[start+len(motif):], "structure": ("F"*seq_len)[:start] + ("S"*len(motif)) + ("F"*seq_len)[start+len(motif):], "label": 0})
    for _ in range(3000): 
        records.append({"sequence": random_sequence_without_motif(seq_len, motif, rng), "structure": "F"*seq_len, "label": 0})

    rng.shuffle(records)
    labels, indices = np.array([r["label"] for r in records], dtype=np.int32), np.arange(len(records))
    train_idx, temp_idx = train_test_split(indices, train_size=8000, random_state=config["seed"], stratify=labels, shuffle=True)
    val_rel, test_rel = train_test_split(np.arange(len(temp_idx)), train_size=2000, random_state=config["seed"], stratify=labels[temp_idx], shuffle=True)
    
    def pack(idx_subset):
        subset = [records[i] for i in idx_subset]
        return {"sequences": [r["sequence"] for r in subset], "structures": [r["structure"] for r in subset], "labels": np.array([r["label"] for r in subset], dtype=np.int32)}
    
    train_split, val_split, test_split = pack(train_idx), pack(temp_idx[val_rel]), pack(temp_idx[test_rel])
    
    result = {
        "x_seq_train": encode_sequence_list(train_split["sequences"], seq_len, config), "x_seq_val": encode_sequence_list(val_split["sequences"], seq_len, config), "x_seq_test": encode_sequence_list(test_split["sequences"], seq_len, config),
        "x_struct_train": np.stack([encode_rnashape_structure(s, config) for s in train_split["structures"]], axis=0), "x_struct_val": np.stack([encode_rnashape_structure(s, config) for s in val_split["structures"]], axis=0), "x_struct_test": np.stack([encode_rnashape_structure(s, config) for s in test_split["structures"]], axis=0),
        "y_train": train_split["labels"], "y_val": val_split["labels"], "y_test": test_split["labels"],
        "train_headers": np.array([f"synthetic_train_{i}" for i in range(len(train_split["labels"]))], dtype=object), "val_headers": np.array([f"synthetic_val_{i}" for i in range(len(val_split["labels"]))], dtype=object), "test_headers": np.array([f"synthetic_test_{i}" for i in range(len(test_split["labels"]))], dtype=object),
        "train_sequences": train_split["sequences"], "val_sequences": val_split["sequences"], "test_sequences": test_split["sequences"]
    }
    np.savez_compressed(synthetic_cache_path, **result)
    return result
    
def filter_summary_to_active_models(df):
    if df.empty or "model_name" not in df.columns: return df
    
    # Add your DeepBind and Attention models to this allowed list!
    allowed_models = {
        "paper_baseline_oli_linear_svm", 
        "paper_baseline_graphprot", 
        "main_ideeps_like_sequence_structure_bilstm",
        "paper_baseline_deepbind", 
        "novel_attention_ideeps", 
        "novel_cross_attention_ideeps" ,
        "novel_moe_swiglu_ideeps"
    }
    
    return df[df["model_name"].isin(allowed_models)].copy()
def append_or_replace_summary_row(summary_csv, row_dict):
    SUMMARY_COLUMNS = ["dataset_name", "model_name", "feature_type", "accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc", "train_size", "val_size", "test_size"]
    if summary_csv.exists(): df = filter_summary_to_active_models(pd.read_csv(summary_csv))
    else: df = pd.DataFrame(columns=SUMMARY_COLUMNS)
    if not df.empty: df = df.loc[~((df["dataset_name"] == row_dict["dataset_name"]) & (df["model_name"] == row_dict["model_name"]))].copy()
    df = pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)
    df.to_csv(summary_csv, index=False)


# --- Initialize Everything ---
set_seed(config["seed"])
paths = make_paths(PROJECT_ROOT)
if not paths["summary_csv"].exists(): pd.DataFrame(columns=["dataset_name", "model_name", "feature_type", "accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc", "train_size", "val_size", "test_size"]).to_csv(paths["summary_csv"], index=False)

print_header("Building real dataset caches")
real_data = {dataset_name: build_processed_dataset(dataset_name, paths, config) for dataset_name in DEFAULT_DATASETS}

print_header("Injecting Synthetic Dataset")
real_data["synthetic"] = build_synthetic_dataset(paths, config)

print("\nSetup complete! Variables are in memory.")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.

# Benchmark Model - 1

In [7]:
from sklearn.svm import LinearSVC
import time

KMER_K = 4
KMER_VOCAB = ["".join(chars) for chars in itertools.product("ACGU", repeat=KMER_K)]
KMER_TO_INDEX = {kmer: idx for idx, kmer in enumerate(KMER_VOCAB)}

def sequence_to_kmer_features(seq):
    seq = seq.upper().replace("T", "U")
    vec = np.zeros(len(KMER_VOCAB), dtype=np.float32)
    total_valid = 0
    for i in range(len(seq) - KMER_K + 1):
        kmer = seq[i : i + KMER_K]
        if all(ch in "ACGU" for ch in kmer): vec[KMER_TO_INDEX[kmer]] += 1.0; total_valid += 1
    if total_valid > 0: vec /= total_valid
    return vec

def run_oli_baseline(dataset_name, data, paths):
    print(f"\nRunning Oli baseline on {dataset_name}")
    start_time = time.time()
    
    X_train = np.stack([sequence_to_kmer_features(s) for s in data["train_sequences"]], axis=0)
    X_val = np.stack([sequence_to_kmer_features(s) for s in data["val_sequences"]], axis=0)
    X_test = np.stack([sequence_to_kmer_features(s) for s in data["test_sequences"]], axis=0)
    
    model = LinearSVC(class_weight="balanced", dual=False, random_state=config["seed"], max_iter=2000)
    model.fit(X_train, data["y_train"])
    
    test_prob = get_positive_probabilities(model, X_test)
    test_metrics = compute_binary_metrics(data["y_test"], test_prob)
    
    save_roc_plot(data["y_test"], test_prob, paths["oli_fig_dir"] / f"{dataset_name}_roc.png", f"{dataset_name} - Oli ROC")
    save_confusion_matrix_plot(test_metrics["confusion_matrix"], paths["oli_fig_dir"] / f"{dataset_name}_confusion_matrix.png", f"{dataset_name} - Oli Confusion Matrix")
    
    pd.DataFrame({"y_true": np.asarray(data["y_test"]).astype(int), "y_prob": test_prob, "y_pred": test_metrics["y_pred"]}).to_csv(paths["oli_pred_dir"] / f"{dataset_name}_paper_baseline_oli_linear_svm_predictions.csv", index=False)
    
    append_or_replace_summary_row(paths["summary_csv"], {
        "dataset_name": dataset_name, "model_name": "paper_baseline_oli_linear_svm", "feature_type": "oli_tetranucleotide_composition",
        "accuracy": test_metrics["accuracy"], "precision": test_metrics["precision"], "recall": test_metrics["recall"],
        "f1_score": test_metrics["f1_score"], "roc_auc": test_metrics["roc_auc"], "pr_auc": test_metrics["pr_auc"],
        "train_size": len(data["y_train"]), "val_size": len(data["y_val"]), "test_size": len(data["y_test"]),
    })
    print(f"Test ROC-AUC      : {round(test_metrics['roc_auc'], 4)} (Took {round(time.time() - start_time, 1)} seconds)")

print_header("Running Oli baseline")
for dataset_name, data in real_data.items():
    run_oli_baseline(dataset_name, data, paths)


Running Oli baseline

Running Oli baseline on 17_ICLIP_HNRNPC_hg19
Test ROC-AUC      : 0.9427 (Took 4.3 seconds)

Running Oli baseline on 10_PARCLIP_ELAVL1A_hg19
Test ROC-AUC      : 0.8589 (Took 4.4 seconds)

Running Oli baseline on 28_ICLIP_TIA1_hg19
Test ROC-AUC      : 0.8882 (Took 4.3 seconds)

Running Oli baseline on synthetic
Test ROC-AUC      : 0.7209 (Took 1.6 seconds)


# Benchmark Model - 2

In [8]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

print("Running Smart DeepBind Patcher...")
deepbind_pred_dir = paths["predictions_dir"] / "paper_baseline_deepbind"
deepbind_model_dir = paths["models_dir"] / "paper_baseline_deepbind"
deepbind_fig_dir = paths["figures_dir"] / "paper_baseline_deepbind" # <-- ADDED FIGURE DIR

deepbind_pred_dir.mkdir(parents=True, exist_ok=True)
deepbind_model_dir.mkdir(parents=True, exist_ok=True)
deepbind_fig_dir.mkdir(parents=True, exist_ok=True) # <-- CREATE FIGURE DIR

for dataset_name, data in real_data.items():
    model_file_path = deepbind_model_dir / f"{dataset_name}_deepbind.keras"
    
    # 1. Load or Train
    if USE_CACHE_DEEPBIND and model_file_path.exists():
        print(f"[{dataset_name}] Found cached model! Loading...")
        model = tf.keras.models.load_model(str(model_file_path))
    else:
        if not USE_CACHE_DEEPBIND:
            print(f"[{dataset_name}] Cache switch OFF. Forcing new DeepBind training on GPU...")
        else:
            print(f"[{dataset_name}] No model found. Training instantly on GPU...")
            
        seq_input = layers.Input(shape=(config["conv_input_length"], data["x_seq_train"].shape[2]))
        x = layers.Conv1D(filters=16, kernel_size=10, activation="relu")(seq_input)
        x = layers.MaxPooling1D(pool_size=3)(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Flatten()(x)
        x = layers.Dense(32, activation="relu")(x)
        output = layers.Dense(1, activation="sigmoid")(x)
        
        model = models.Model(inputs=seq_input, outputs=output)
        model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=config["learning_rate"]),loss="binary_crossentropy",metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),tf.keras.metrics.AUC(name="auc"),],)

        
        history = model.fit(
            data["x_seq_train"], data["y_train"], 
            validation_data=(data["x_seq_val"], data["y_val"]), 
            epochs=config["max_epochs"], 
            batch_size=config["batch_size"], 
            verbose=0, 
            callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
        )
        model.save(str(model_file_path))
        
        # Save training history if trained from scratch
        save_training_history_plot(history, deepbind_fig_dir / f"{dataset_name}_history.png", f"{dataset_name} - DeepBind")
    
    # 2. Score
    test_prob = model.predict(data["x_seq_test"], verbose=0).ravel()
    
    # <-- ADDED: Calculate metrics and save ROC/Confusion Matrix figures -->
    test_metrics = compute_binary_metrics(data["y_test"], test_prob)
    save_roc_plot(data["y_test"], test_prob, deepbind_fig_dir / f"{dataset_name}_roc.png", f"{dataset_name} - DeepBind ROC")
    save_confusion_matrix_plot(test_metrics["confusion_matrix"], deepbind_fig_dir / f"{dataset_name}_confusion_matrix.png", f"{dataset_name} - DeepBind CM")
    
    # 3. Save Predictions
    pd.DataFrame({"y_true": data["y_test"], "y_prob": test_prob}).to_csv(deepbind_pred_dir / f"{dataset_name}_paper_baseline_deepbind_predictions.csv", index=False)
    
    # 4. Force into CSV
    append_or_replace_summary_row(paths["summary_csv"], {
        "dataset_name": dataset_name, 
        "model_name": "paper_baseline_deepbind", 
        "feature_type": "sequence_cnn_only",
        "accuracy": test_metrics["accuracy"], 
        "precision": test_metrics["precision"], 
        "recall": test_metrics["recall"], 
        "f1_score": test_metrics["f1_score"],
        "roc_auc": test_metrics["roc_auc"], 
        "pr_auc": test_metrics["pr_auc"],
        "train_size": len(data["y_train"]), "val_size": len(data["y_val"]), "test_size": len(data["y_test"])
    })
    print(f"[{dataset_name}] Successfully locked into CSV and saved figures!")

print("\nAll 4 DeepBind models are securely in the CSV. RUN CELL 5 NOW.")

Running Smart DeepBind Patcher...
[17_ICLIP_HNRNPC_hg19] Cache switch OFF. Forcing new DeepBind training on GPU...


I0000 00:00:1776037474.469426      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776037474.472083      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1776037477.307947    1626 service.cc:152] XLA service 0x7c3760006170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776037477.307981    1626 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776037477.307985    1626 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776037477.714959    1626 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776037480.638471    1626 device_compiler.h:188] Compiled clust

[17_ICLIP_HNRNPC_hg19] Successfully locked into CSV and saved figures!
[10_PARCLIP_ELAVL1A_hg19] Cache switch OFF. Forcing new DeepBind training on GPU...
[10_PARCLIP_ELAVL1A_hg19] Successfully locked into CSV and saved figures!
[28_ICLIP_TIA1_hg19] Cache switch OFF. Forcing new DeepBind training on GPU...
[28_ICLIP_TIA1_hg19] Successfully locked into CSV and saved figures!
[synthetic] Cache switch OFF. Forcing new DeepBind training on GPU...
[synthetic] Successfully locked into CSV and saved figures!

All 4 DeepBind models are securely in the CSV. RUN CELL 5 NOW.


# iDeep-s like Model

In [ ]:
def build_ideeps_like_model(config, sequence_channels=4, structure_channels=6):
    seq_input = layers.Input(shape=(config["conv_input_length"], sequence_channels), name="sequence_input")
    struct_input = layers.Input(shape=(config["conv_input_length"], structure_channels), name="structure_input")

    seq_branch = layers.Dropout(0.5)(layers.MaxPooling1D(pool_size=3)(layers.Conv1D(filters=16, kernel_size=10, activation="relu")(seq_input)))
    struct_branch = layers.Dropout(0.5)(layers.MaxPooling1D(pool_size=3)(layers.Conv1D(filters=16, kernel_size=10, activation="relu")(struct_input)))

    merged = layers.Dense(32, activation="relu")(layers.Dropout(0.10)(layers.Bidirectional(layers.LSTM(16, return_sequences=False))(layers.Concatenate(axis=-1)([seq_branch, struct_branch]))))
    model = models.Model(inputs=[seq_input, struct_input], outputs=layers.Dense(1, activation="sigmoid")(merged), name="ideeps_like_model")
    
    model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=config["learning_rate"]), loss="binary_crossentropy", metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")])
    return model

def run_main_model(dataset_name, data, paths, config):
    print(f"\nRunning iDeepS-like model on {dataset_name}")
    
    model_file_path = paths["main_model_dir"] / f"{dataset_name}_best.keras"
    
    if USE_CACHE_IDEEPS and model_file_path.exists():
        print(f"--- SKIP: Cached iDeepS model found! Loading {dataset_name} ---")
        model = tf.keras.models.load_model(str(model_file_path))
    else:
        if not USE_CACHE_IDEEPS:
            print(f"--- Cache switch OFF: Forcing new iDeepS training for {dataset_name} ---")
        else:
            print(f"--- Training new iDeepS model ---")
            
        model = build_ideeps_like_model(config, sequence_channels=data["x_seq_train"].shape[2], structure_channels=data["x_struct_train"].shape[2])

        callbacks = [
            EarlyStopping(monitor="val_loss", patience=config["early_stopping_patience"], restore_best_weights=True, verbose=1),
            ModelCheckpoint(filepath=str(model_file_path), monitor="val_loss", save_best_only=True, verbose=0),
        ]

        history = model.fit([data["x_seq_train"], data["x_struct_train"]], data["y_train"], validation_data=([data["x_seq_val"], data["x_struct_val"]], data["y_val"]), epochs=config["max_epochs"], batch_size=config["batch_size"], verbose=1, callbacks=callbacks)
        save_training_history_plot(history, paths["main_fig_dir"] / f"{dataset_name}_history.png", f"{dataset_name} - iDeepS-like model")

    test_prob = model.predict([data["x_seq_test"], data["x_struct_test"]], verbose=0).ravel()
    test_metrics = compute_binary_metrics(data["y_test"], test_prob)

    save_roc_plot(data["y_test"], test_prob, paths["main_fig_dir"] / f"{dataset_name}_roc.png", f"{dataset_name} - iDeepS-like ROC")
    save_confusion_matrix_plot(test_metrics["confusion_matrix"], paths["main_fig_dir"] / f"{dataset_name}_confusion_matrix.png", f"{dataset_name} - iDeepS-like Confusion Matrix")

    pd.DataFrame({"y_true": np.asarray(data["y_test"]).astype(int), "y_prob": test_prob, "y_pred": test_metrics["y_pred"]}).to_csv(paths["main_pred_dir"] / f"{dataset_name}_ideeps_like_predictions.csv", index=False)

    append_or_replace_summary_row(paths["summary_csv"], {
        "dataset_name": dataset_name, "model_name": "main_ideeps_like_sequence_structure_bilstm", "feature_type": "sequence_plus_structure",
        "accuracy": test_metrics["accuracy"], "precision": test_metrics["precision"], "recall": test_metrics["recall"],
        "f1_score": test_metrics["f1_score"], "roc_auc": test_metrics["roc_auc"], "pr_auc": test_metrics["pr_auc"],
        "train_size": len(data["y_train"]), "val_size": len(data["y_val"]), "test_size": len(data["y_test"]),
    })
    print("Test ROC-AUC      :", round(test_metrics["roc_auc"], 4))

print_header("Running main iDeepS-like model")
for dataset_name, data in real_data.items():
    run_main_model(dataset_name, data, paths, config)

# Novel Model - 1 Cross-Attention

In [9]:
def build_cross_attention_model(config, sequence_channels=4, structure_channels=6):
    seq_input = layers.Input(shape=(config["conv_input_length"], sequence_channels), name="seq_input")
    struct_input = layers.Input(shape=(config["conv_input_length"], structure_channels), name="struct_input")
    
    # 1. Convolutions (padding="same" keeps temporal dimensions aligned for attention)
    seq_conv = layers.Conv1D(filters=16, kernel_size=10, activation="relu", padding="same")(seq_input)
    struct_conv = layers.Conv1D(filters=16, kernel_size=10, activation="relu", padding="same")(struct_input)
    
    # 2. Cross-Attention: "Which structural contexts are most relevant to this specific sequence motif?"
    attention_out = layers.Attention(name="seq_to_struct_attention")([seq_conv, struct_conv])
    
    # 3. Merge original sequence features with the attended structural context
    merged_features = layers.Concatenate(axis=-1)([seq_conv, attention_out])
    
    # 4. Apply Max Pooling AFTER attention to preserve local geometry during the attention calculation
    pooled_features = layers.MaxPooling1D(pool_size=3)(merged_features)
    pooled_features = layers.Dropout(0.5)(pooled_features)
    
    # 5. Pass into BLSTM
    lstm_out = layers.Bidirectional(layers.LSTM(16, return_sequences=False))(pooled_features)
    lstm_out = layers.Dropout(0.10)(lstm_out)
    
    x = layers.Dense(32, activation="relu")(lstm_out)
    output = layers.Dense(1, activation="sigmoid")(x)
    
    model = models.Model(inputs=[seq_input, struct_input], outputs=output, name="novel_cross_attention_ideeps")
    model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=config["learning_rate"]),loss="binary_crossentropy",metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),tf.keras.metrics.AUC(name="auc"),],)
    return model

print("Running NOVELTY: Cross-Attention iDeepS...")
attention_pred_dir = ensure_dir(paths["predictions_dir"] / "novel_cross_attention_ideeps")
attention_model_dir = ensure_dir(paths["models_dir"] / "novel_cross_attention_ideeps")
attention_fig_dir = ensure_dir(paths["figures_dir"] / "novel_cross_attention_ideeps") 

for dataset_name, data in real_data.items():
    model_file_path = attention_model_dir / f"{dataset_name}_cross_attention.keras"
    
    if USE_CACHE_ATTENTION and model_file_path.exists():
        print(f"[{dataset_name}] Found cached Cross-Attention model! Loading...")
        model = tf.keras.models.load_model(str(model_file_path))
    else:
        print(f"Training Cross-Attention iDeepS on {dataset_name}...")
        model = build_cross_attention_model(config, sequence_channels=data["x_seq_train"].shape[2], structure_channels=data["x_struct_train"].shape[2])
        
        history = model.fit(
            {"seq_input": data["x_seq_train"], "struct_input": data["x_struct_train"]}, data["y_train"], 
            validation_data=({"seq_input": data["x_seq_val"], "struct_input": data["x_struct_val"]}, data["y_val"]), 
            epochs=config["max_epochs"], batch_size=config["batch_size"], verbose=1, 
            callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
        )
        model.save(str(model_file_path))
        save_training_history_plot(history, attention_fig_dir / f"{dataset_name}_history.png", f"{dataset_name} - Cross-Attention")
    
    test_prob = model.predict({"seq_input": data["x_seq_test"], "struct_input": data["x_struct_test"]}, verbose=0).ravel()
    test_metrics = compute_binary_metrics(data["y_test"], test_prob)
    
    save_roc_plot(data["y_test"], test_prob, attention_fig_dir / f"{dataset_name}_roc.png", f"{dataset_name} - Cross-Attention ROC")
    save_confusion_matrix_plot(test_metrics["confusion_matrix"], attention_fig_dir / f"{dataset_name}_confusion_matrix.png", f"{dataset_name} - Cross-Attention CM")
    
    pd.DataFrame({"y_true": data["y_test"], "y_prob": test_prob}).to_csv(attention_pred_dir / f"{dataset_name}_novel_cross_attention_predictions.csv", index=False)
    
    append_or_replace_summary_row(paths["summary_csv"], {
        "dataset_name": dataset_name, "model_name": "novel_cross_attention_ideeps", "feature_type": "sequence_cross_structure",
        "accuracy": test_metrics['accuracy'], "precision": test_metrics['precision'], "recall": test_metrics['recall'], "f1_score": test_metrics['f1_score'],
        "roc_auc": test_metrics['roc_auc'], "pr_auc": test_metrics['pr_auc'],
        "train_size": len(data["y_train"]), "val_size": len(data["y_val"]), "test_size": len(data["y_test"])
    })

print("Cross-Attention Model Finished! Run Cell 7 to plot.")

Running NOVELTY: Cross-Attention iDeepS...
Training Cross-Attention iDeepS on 17_ICLIP_HNRNPC_hg19...
Epoch 1/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8216 - auc: 0.7715 - loss: 0.4051 - val_accuracy: 0.9052 - val_auc: 0.9574 - val_loss: 0.2305
Epoch 2/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9116 - auc: 0.9536 - loss: 0.2133 - val_accuracy: 0.9180 - val_auc: 0.9683 - val_loss: 0.1993
Epoch 3/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9252 - auc: 0.9638 - loss: 0.1871 - val_accuracy: 0.9297 - val_auc: 0.9736 - val_loss: 0.1750
Epoch 4/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9283 - auc: 0.9691 - loss: 0.1747 - val_accuracy: 0.9193 - val_auc: 0.9737 - val_loss: 0.1969
Epoch 5/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9322 - auc: 0.9714 - loss: 0.1688 - val_accuracy: 0.9340 - val_auc: 0.9768 - val_loss: 0.1675
Epoch 6/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9339 - auc: 0.9732 - loss: 0.1

# Novel Model 2 - MoE + SwiGLU Model

In [10]:
def build_moe_swiglu_model(config, sequence_channels=4, structure_channels=6):
    seq_input = layers.Input(shape=(config["conv_input_length"], sequence_channels), name="seq_input")
    struct_input = layers.Input(shape=(config["conv_input_length"], structure_channels), name="struct_input")
    
    # 1. Base Convolutions
    seq_conv = layers.Conv1D(filters=16, kernel_size=10, activation="relu", padding="same")(seq_input)
    struct_conv = layers.Conv1D(filters=16, kernel_size=10, activation="relu", padding="same")(struct_input)
    
    merged_features = layers.Concatenate(axis=-1)([seq_conv, struct_conv])
    pooled_features = layers.MaxPooling1D(pool_size=3)(merged_features)
    pooled_features = layers.Dropout(0.5)(pooled_features)
    
    # 2. BLSTM
    lstm_out = layers.Bidirectional(layers.LSTM(16, return_sequences=False))(pooled_features)
    lstm_out = layers.Dropout(0.10)(lstm_out)
    
    # 3. Structure-driven Router for MoE
    # We use the raw structural features to decide which expert handles this RNA
    struct_pooled = layers.GlobalMaxPooling1D()(struct_conv)
    num_experts = 4
    router_weights = layers.Dense(num_experts, activation="softmax", name="expert_router")(struct_pooled)
    
    # 4. SwiGLU Experts
    expert_outputs = []
    for i in range(num_experts):
        # SwiGLU implementation: Dense(x) * Swish(Dense(x))
        gate = layers.Dense(32, activation="swish")(lstm_out)
        value = layers.Dense(32)(lstm_out)
        expert_out = layers.Multiply()([gate, value])
        
        # Reshape to (batch, 1, 32) so we can stack them
        expert_outputs.append(layers.Reshape((1, 32))(expert_out))
        
    # 5. Combine Experts using Router Weights
    all_experts = layers.Concatenate(axis=1)(expert_outputs) # (batch, num_experts, 32)
    router_reshaped = layers.Reshape((1, num_experts))(router_weights) # (batch, 1, num_experts)
    
    # Weighted sum: (batch, 1, num_experts) * (batch, num_experts, 32) -> (batch, 1, 32)
    moe_out = layers.Dot(axes=(2, 1))([router_reshaped, all_experts])
    moe_out = layers.Flatten()(moe_out)
    
    output = layers.Dense(1, activation="sigmoid")(moe_out)
    
    model = models.Model(inputs=[seq_input, struct_input], outputs=output, name="novel_moe_swiglu_ideeps")
    
    # ---> ADDED METRICS HERE TO PRINT DURING TRAINING <---
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=config["learning_rate"]), 
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

print("Running NOVELTY: Structure-Routed MoE with SwiGLU...")
attention_pred_dir = ensure_dir(paths["predictions_dir"] / "novel_moe_swiglu_ideeps")
attention_model_dir = ensure_dir(paths["models_dir"] / "novel_moe_swiglu_ideeps")
attention_fig_dir = ensure_dir(paths["figures_dir"] / "novel_moe_swiglu_ideeps") 

for dataset_name, data in real_data.items():
    model_file_path = attention_model_dir / f"{dataset_name}_moe_swiglu.keras"
    
    if USE_CACHE_ATTENTION and model_file_path.exists():
        print(f"[{dataset_name}] Found cached MoE model! Loading...")
        model = tf.keras.models.load_model(str(model_file_path))
    else:
        print(f"Training MoE-SwiGLU on {dataset_name}...")
        model = build_moe_swiglu_model(config, sequence_channels=data["x_seq_train"].shape[2], structure_channels=data["x_struct_train"].shape[2])
        
        history = model.fit(
            {"seq_input": data["x_seq_train"], "struct_input": data["x_struct_train"]}, data["y_train"], 
            validation_data=({"seq_input": data["x_seq_val"], "struct_input": data["x_struct_val"]}, data["y_val"]), 
            epochs=config["max_epochs"], batch_size=config["batch_size"], verbose=1, 
            callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
        )
        model.save(str(model_file_path))
        save_training_history_plot(history, attention_fig_dir / f"{dataset_name}_history.png", f"{dataset_name} - MoE SwiGLU")
    
    test_prob = model.predict({"seq_input": data["x_seq_test"], "struct_input": data["x_struct_test"]}, verbose=0).ravel()
    test_metrics = compute_binary_metrics(data["y_test"], test_prob)
    
    save_roc_plot(data["y_test"], test_prob, attention_fig_dir / f"{dataset_name}_roc.png", f"{dataset_name} - MoE SwiGLU ROC")
    save_confusion_matrix_plot(test_metrics["confusion_matrix"], attention_fig_dir / f"{dataset_name}_confusion_matrix.png", f"{dataset_name} - MoE SwiGLU CM")
    
    pd.DataFrame({"y_true": data["y_test"], "y_prob": test_prob}).to_csv(attention_pred_dir / f"{dataset_name}_novel_moe_swiglu_predictions.csv", index=False)
    
    append_or_replace_summary_row(paths["summary_csv"], {
        "dataset_name": dataset_name, "model_name": "novel_moe_swiglu_ideeps", "feature_type": "moe_swiglu_structure_routed",
        "accuracy": test_metrics['accuracy'], "precision": test_metrics['precision'], "recall": test_metrics['recall'], "f1_score": test_metrics['f1_score'],
        "roc_auc": test_metrics['roc_auc'], "pr_auc": test_metrics['pr_auc'],
        "train_size": len(data["y_train"]), "val_size": len(data["y_val"]), "test_size": len(data["y_test"])
    })

print("MoE-SwiGLU Model Finished! Run Cell 7 to plot.")

Running NOVELTY: Structure-Routed MoE with SwiGLU...
Training MoE-SwiGLU on 17_ICLIP_HNRNPC_hg19...
Epoch 1/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8128 - auc: 0.7175 - loss: 0.4456 - val_accuracy: 0.8893 - val_auc: 0.9531 - val_loss: 0.2448
Epoch 2/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9081 - auc: 0.9434 - loss: 0.2322 - val_accuracy: 0.9195 - val_auc: 0.9673 - val_loss: 0.1976
Epoch 3/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9260 - auc: 0.9600 - loss: 0.1941 - val_accuracy: 0.9207 - val_auc: 0.9715 - val_loss: 0.1850
Epoch 4/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9286 - auc: 0.9659 - loss: 0.1818 - val_accuracy: 0.9155 - val_auc: 0.9732 - val_loss: 0.1922
Epoch 5/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9337 - auc: 0.9665 - loss: 0.1766 - val_accuracy: 0.9275 - val_auc: 0.9751 - val_loss: 0.1750
Epoch 6/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9325 - auc: 0.9713 - loss: 0

# Final Results

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

print_header("Final Metrics & Comparison Outputs")

df = pd.read_csv(paths["summary_csv"])

# 1. Map ALL 5 models to clean display names
name_map = {
    "paper_baseline_oli_linear_svm": "Benchmark 1: Oli", 
    "paper_baseline_deepbind": "Benchmark 2: DeepBind", 
    "main_ideeps_like_sequence_structure_bilstm": "Main model: iDeepS",
    "novel_cross_attention_ideeps": "NOVELTY 1: Cross-Attention",
    "novel_moe_swiglu_ideeps": "NOVELTY 2: MoE-SwiGLU" 
}

df["friendly_model_name"] = df["model_name"].map(name_map)
df = df.dropna(subset=["friendly_model_name"]).copy()

# 2. Sort for clean output (1 through 5)
df["model_rank"] = df["friendly_model_name"].map({
    "Benchmark 1: Oli": 1, 
    "Benchmark 2: DeepBind": 2, 
    "Main model: iDeepS": 3, 
    "NOVELTY 1: Cross-Attention": 4,
    "NOVELTY 2: MoE-SwiGLU": 5
})
df = df.sort_values(by=["dataset_name", "model_rank"]).drop(columns=["model_rank"])

print("=========================================================")
print("FINAL METRICS TABLE:")
print("=========================================================")
display(df[["dataset_name", "friendly_model_name", "accuracy", "f1_score", "roc_auc"]])

# 3. Plot the final 5-Way ROC Curves
datasets = df["dataset_name"].unique()
colors = {
    "Benchmark 1: Oli": "gray",
    "Benchmark 2: DeepBind": "blue",
    "Main model: iDeepS": "green",
    "NOVELTY 1: Cross-Attention": "orange",
    "NOVELTY 2: MoE-SwiGLU": "red"
}

for dataset_name in datasets:
    plt.figure(figsize=(8, 6))
    dataset_df = df[df["dataset_name"] == dataset_name]
    
    for _, row in dataset_df.iterrows():
        model_id = row["model_name"]
        friendly_name = row["friendly_model_name"]
        
        # Route to the correct prediction CSV for all 5 architectures
        if model_id == "paper_baseline_oli_linear_svm":
            pred_path = paths["oli_pred_dir"] / f"{dataset_name}_paper_baseline_oli_linear_svm_predictions.csv"
        elif model_id == "paper_baseline_deepbind":
            pred_path = paths["predictions_dir"] / "paper_baseline_deepbind" / f"{dataset_name}_paper_baseline_deepbind_predictions.csv"
        elif model_id == "novel_cross_attention_ideeps":
            pred_path = paths["predictions_dir"] / "novel_cross_attention_ideeps" / f"{dataset_name}_novel_cross_attention_predictions.csv"
        elif model_id == "novel_moe_swiglu_ideeps":
            pred_path = paths["predictions_dir"] / "novel_moe_swiglu_ideeps" / f"{dataset_name}_novel_moe_swiglu_predictions.csv"
        else:
            pred_path = paths["main_pred_dir"] / f"{dataset_name}_ideeps_like_predictions.csv"
            
        if pred_path.exists():
            preds = pd.read_csv(pred_path)
            fpr, tpr, _ = roc_curve(preds["y_true"], preds["y_prob"])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors.get(friendly_name, "black"), lw=2, label=f"{friendly_name} (AUC = {roc_auc:.4f})")
            
    plt.plot([0, 1], [0, 1], 'k--', lw=1.5)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curve Comparison - {dataset_name}', fontsize=14)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(alpha=0.3)
    
    plot_path = paths["final_comparison_fig_dir"] / f"{dataset_name}_final_roc_comparison.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

print("All plots saved to /figures/final_comparison/")


Final Metrics & Comparison Outputs
FINAL METRICS TABLE:


,dataset_name,friendly_model_name,accuracy,f1_score,roc_auc
5,10_PARCLIP_ELAVL1A_hg19,Benchmark 1: Oli,0.8028,0.619305,0.858936
9,10_PARCLIP_ELAVL1A_hg19,Benchmark 2: DeepBind,0.8934,0.711892,0.894828
1,10_PARCLIP_ELAVL1A_hg19,Main model: iDeepS,0.8940,0.713823,0.893132
13,10_PARCLIP_ELAVL1A_hg19,NOVELTY 1: Cross-Attention,0.8947,0.725280,0.894378
17,10_PARCLIP_ELAVL1A_hg19,NOVELTY 2: MoE-SwiGLU,0.8937,0.721801,0.893132
4,17_ICLIP_HNRNPC_hg19,Benchmark 1: Oli,0.8820,0.746018,0.942670
8,17_ICLIP_HNRNPC_hg19,Benchmark 2: DeepBind,0.9363,0.840949,0.974616
0,17_ICLIP_HNRNPC_hg19,Main model: iDeepS,0.9375,0.848778,0.976559
12,17_ICLIP_HNRNPC_hg19,NOVELTY 1: Cross-Attention,0.9321,0.839214,0.974755
16,17_ICLIP_HNRNPC_hg19,NOVELTY 2: MoE-SwiGLU,0.9260,0.830508,0.975077


All plots saved to /figures/final_comparison/


In [12]:
!zip -r /kaggle/working/my_final_ideeps_project.zip /kaggle/working/ideeps_output/

  adding: kaggle/working/ideeps_output/ (stored 0%)
  adding: kaggle/working/ideeps_output/models/ (stored 0%)
  adding: kaggle/working/ideeps_output/models/ideeps_like_model/ (stored 0%)
  adding: kaggle/working/ideeps_output/models/ideeps_like_model/17_ICLIP_HNRNPC_hg19_best.keras (deflated 47%)
  adding: kaggle/working/ideeps_output/models/ideeps_like_model/28_ICLIP_TIA1_hg19_best.keras (deflated 47%)
  adding: kaggle/working/ideeps_output/models/ideeps_like_model/10_PARCLIP_ELAVL1A_hg19_best.keras (deflated 47%)
  adding: kaggle/working/ideeps_output/models/ideeps_like_model/synthetic_best.keras (deflated 47%)
  adding: kaggle/working/ideeps_output/models/novel_cross_attention_ideeps/ (stored 0%)
  adding: kaggle/working/ideeps_output/models/novel_cross_attention_ideeps/10_PARCLIP_ELAVL1A_hg19_cross_attention.keras (deflated 46%)
  adding: kaggle/working/ideeps_output/models/novel_cross_attention_ideeps/28_ICLIP_TIA1_hg19_cross_attention.keras (deflated 46%)
  adding: kaggle/workin